# Chunking

In [1]:
import json
file = "/Users/joshturner/Desktop/LTI Project/MathBot/JSON/lesson1.json"
with open(file, "r") as f:
    data = json.load(f)

chunks = []

#
# Learning objectives
#
for obj in data["learning_objectives"]:
    chunks.append({
        "id": f"week{data['week']}_objective_{obj['tag']}",
        "text": obj["description"],
        "metadata": {
            "week": data["week"],
            "type": "learning_objective",
            "tag": obj["tag"]
        }
    })

#
# Discussions
#
for i, discussion in enumerate(data["contents"]["other_material"]):
    chunks.append({
        "id": f"week{data['week']}_discussion_{i}",
        "text": discussion["content_plain"],
        "metadata": {
            "week": data["week"],
            "type": discussion["type"]
        }
    })

#
# Problems + subproblems
#
for problem in data["contents"]["problems"]:

    context = problem["context_plain"]

    for sub in problem["subproblems"]:

        question = sub["plain_text"]["question"]
        answer = sub["plain_text"]["answer"]

        chunk_text = f"""
Problem: {problem['name']}

Context:
{context}

Question:
{question}

Answer:
{answer}
"""

        chunks.append({
            "id": f"week{data['week']}_{problem['name']}_{sub['part']}",
            "text": chunk_text,
            "metadata": {
                "week": data["week"],
                "type": "subproblem",
                "problem_name": problem["name"],
                "part": sub["part"],
                "learning_tag": problem["learning_tag"],
                "keywords": problem["keywords"]
            }
        })

print(f"Created {len(chunks)} chunks")

Created 8 chunks


# Sentence Transformer

In [4]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)

embedding = model.encode(chunks[0]["text"])

/Users/joshturner/.pyenv/versions/calculus-rag/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 13192.76it/s]


# Vector DB

In [5]:
import chromadb
from sentence_transformers import SentenceTransformer

client = chromadb.Client()

collection = client.create_collection(
    name="calculus_notes"
)

model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)

for chunk in chunks:
    embedding = model.encode(chunk["text"]).tolist()

    collection.add(
        ids=[chunk["id"]],
        documents=[chunk["text"]],
        embeddings=[embedding],
        metadatas=[chunk["metadata"]]
    )

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8809.98it/s]


# Query the Vector DB

In [6]:
query = "What does the derivative represent physically?"

query_embedding = model.encode(query).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

print(results["documents"])

[['\nProblem: Exercise A_b\n\nContext:\nThe equation F = 9/5 C + 32 gives the relationship between temperature in degrees Fahrenheit (F) and temperature in degrees celsius (C).\n\nQuestion:\nWhat is the slope of the graph describing the relationship between F and C?\n\nAnswer:\nm = 9/5\n', 'Net change is the difference between two quantities; AROC and slope are mathematically the same; Both slope and AROC are computed by the quotient of net changes', '\nProblem: Exercise A_b\n\nContext:\nThe equation F = 9/5 C + 32 gives the relationship between temperature in degrees Fahrenheit (F) and temperature in degrees celsius (C).\n\nQuestion:\nWhat is the net change in the temperature in degrees Fahrenheit when the temperature in degrees Celsius increases by 1?\n\nAnswer:\nDelta F = 9/5 when Delta C = 1\n']]


# Response from LLM

In [ ]:
import ollama

response = ollama.chat(
    model='qwen3:8b',
    messages=[
        {
            'role': 'user',
            'content': 'What is slope?'
        }
    ]
)

print(response['message']['content'])